In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
import joblib


In [2]:
X = pd.read_csv("../data/processed/secom_features_clean.csv")
print("X shape:", X.shape)
X.head()


X shape: (1567, 582)


,sensor_0,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,...,sensor_580,sensor_581,sensor_582,sensor_583,sensor_584,sensor_585,sensor_586,sensor_587,sensor_588,sensor_589
0,0.224463,0.849523,-0.436430,0.035804,-0.050121,0.0,-0.564354,0.265894,0.509848,1.128455,...,-0.138300,-0.179550,0.118679,-0.204833,-0.093165,-0.197057,-0.077554,-0.190165,-0.238334,-0.295753
1,1.107287,-0.383106,1.016977,0.155282,-0.059585,0.0,0.197639,0.321868,0.457021,0.022620,...,0.516737,2.233265,0.530183,0.406734,0.444748,0.385113,-0.960123,0.411970,0.250272,1.156846
2,-1.114000,0.798901,-0.481447,0.688278,-0.047447,0.0,-0.906768,0.254699,-0.260885,0.327222,...,4.950839,0.008115,-1.262799,0.022320,0.014418,0.029888,2.991195,3.627143,3.321511,-0.178955
3,-0.350156,-0.199072,-0.051705,-1.104376,-0.050831,0.0,0.502662,-0.013974,0.343240,-0.765369,...,-0.289463,-0.151957,-0.322218,-0.292200,-0.362121,-0.283360,-0.101845,-0.178804,-0.308135,-0.275049
4,0.242296,0.087328,1.117227,-0.156616,-0.047033,0.0,-0.115954,0.187531,0.545066,-0.149545,...,-0.138300,-0.179550,-5.906917,26.867221,27.071429,26.913337,-0.101845,-0.178804,-0.308135,-0.275049


In [3]:
iso = IsolationForest(
    n_estimators=200,
    contamination=0.03,   # 3% anomalies flagged
    random_state=42,
    n_jobs=-1
)

iso.fit(X)
print("Model trained.")


Model trained.


In [4]:
# Higher = more anomalous (we invert sklearn's convention)
anomaly_score = -iso.score_samples(X)

scores = pd.DataFrame({
    "anomaly_score": anomaly_score
})

scores.describe()


,anomaly_score
count,1567.000000
mean,0.402750
std,0.016949
min,0.366763
25%,0.391042
50%,0.400411
75%,0.411559
max,0.516371


In [5]:
threshold = np.quantile(scores["anomaly_score"], 0.97)
scores["is_anomaly"] = (scores["anomaly_score"] >= threshold).astype(int)

scores["is_anomaly"].value_counts()


is_anomaly
0    1520
1      47
Name: count, dtype: int64

In [6]:
joblib.dump(iso, "../models/isolation_forest.joblib")
scores.to_csv("../data/processed/secom_anomaly_scores.csv", index=False)

print("Saved model to models/ and scores to data/processed/")


Saved model to models/ and scores to data/processed/
